### Importing Files

In [ ]:
import torch 
import torch_geometric
import numpy as np
import pandas as pd
import scipy
import mne

from torch_geometric.nn import MessagePassing
import torch.nn.functional as F
from torch_geometric.utils import dense_to_sparse, softmax
from torch_geometric.data import Data


### Common declarations

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
montage = mne.channels.make_standard_montage('standard_1020')

## GCN (Trial)

### Basic Execution

In [57]:
X_sample = torch.randn(5,6) # Taking random values for each node's features
labels = torch.tensor([0, 0, 0, 1, 1])

coords = torch.tensor([
    [0.0,  0.7,  0.0],   # Fz
    [0.3,  0.5,  0.0],   # F3
   [-0.3,  0.5,  0.0],   # F4
    [0.0,  0.0,  0.9],   # Cz
    [0.5,  0.0,  0.5],   # C4
])

# Coordinates of that help us choose whether nodes should be considered neighbors or not. 

node_names = ["Fz", "F3", "F4", "CZ", "C4"]

W = torch.randn(6,4) 

A = torch.zeros(5,5)

def check_neighbors():
    for electrode in range(len(coords)): # Running over each electrode's coordinates
        for neighbor in range(len(coords)):
            if neighbor != electrode:
                distance_from_neighbor = 0.0
                for coord in range(len(coords[electrode])): # Running over each coordinate of electrode and the comparison electrode
                    distance_from_neighbor += (coords[electrode][coord] - coords[neighbor][coord]) ** 2
                
                distance_from_neighbor = distance_from_neighbor ** 0.5

            else:
                continue

            # print(f"Distance between: {node_names[electrode]} and {node_names[neighbor]} is: {distance_from_neighbor}")

            if distance_from_neighbor < 0.61:
                print(f"{node_names[electrode]} and {node_names[neighbor]} are neighbours")
            
            # else:
            #     print(f"{node_names[electrode]} and {node_names[neighbor]} are not neighbours")
            #     print(distance_from_neighbor)

### GCN Layer

In [58]:
class GCNLayer(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.W = torch.nn.Parameter(torch.randn(in_features, out_features))
        self.in_features = in_features
        self.out_features = out_features
    
    def forward(self, X, A):
        degrees = A.sum(dim=1)
        D_inv = torch.diag(1.0/degrees)

        normalized_A = torch.matmul(D_inv, A)
        normalized_result = torch.matmul(normalized_A, X)

        transformed_results = torch.matmul(normalized_result, self.W)

        return transformed_results

### Working of GCN layer

In [59]:
#Creating adjacency matrix
diff = coords.unsqueeze(0) - coords.unsqueeze(1) # Converts tensors to single dimensional tensors
dist = (diff ** 2).sum(dim=-1) ** 0.5  
A = (dist < 0.61).float() # Condition for each value in the tensor. if true => 1., if not => 0.
A.fill_diagonal_(0) # Fills diagonals with zeros

# Fixing degree-scale problem
identity_matrix = torch.eye(5)
A_hat = A + identity_matrix

edge_index, _ = dense_to_sparse(A_hat)

In [60]:
gcn = GCNLayer(6, 4)
optimizer_gcn = torch.optim.Adam(gcn.parameters(), lr = 0.01)

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = gcn(X_sample, A_hat)
#     loss = criterion(out, labels)
#     loss.backward()
#     optimizer.step()

    # if epoch % 10 == 0:
    #     print(f"Epoch {epoch}, Loss: {loss.item()}")

## GAT

### GAT Layer

In [61]:
class GATLayer(MessagePassing):
    def __init__(self, in_features, out_features):
        super().__init__(aggr='add')
        self.W = torch.nn.Linear(in_features, out_features, bias = False) #Wraps X @ W into cleaner syntax
        self.a = torch.nn.Parameter(torch.randn(2 * out_features)) # fEATURE vector for the two nodes

    def forward(self, X, edge_index):
        X = self.W(X)
        return self.propagate(edge_index, X=X, size=(X.shape[0], X.shape[0]))
       
    def message(self, X_i, X_j, index):
        cont_msg = torch.cat([X_i, X_j], dim=1)
        att_score = (self.a * cont_msg).sum(dim=-1, keepdim=True)
        final_score = softmax(att_score, index)

        return X_j * final_score

    def update(self, aggr_out):
        return aggr_out

### Working

In [62]:
# gat= GATLayer(6,4)
# optimizer = torch.optim.Adam(gat.parameters(), lr = 0.01)

# for epoch in range(1000):
#     optimizer.zero_grad()
#     out = gat(X_sample, edge_index)
#     loss = criterion(out, labels)
#     loss.backward()
#     optimizer.step()

#     if epoch % 200 == 0:
#         print(f"Epoch {epoch}, Loss: {loss.item()}")

## BrainSync Model

### Single Layer

In [63]:
class BrainSync(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.gat = GATLayer(in_features, out_features)
        self.lin_layer = torch.nn.Linear(out_features, 2)

    def forward(self, X, edge_index):
        out = self.gat(X, edge_index)
        mean = torch.mean(out, dim=0)
        classes = self.lin_layer(mean)

        return classes

In [64]:
# X = torch.randn(64, 6)
# edge_index_final = torch.randint(0, 64, (2, 200))
# labels_final = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

# sync = BrainSync(6, 4)

# optimizer = torch.optim.Adam(sync.parameters(), lr = 0.01)

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = sync(X, edge_index_final)
#     loss = criterion(out, labels_final)
#     loss.backward()
#     optimizer.step()

#     # if epoch % 10 == 0:
#     #     print(f"Epoch {epoch}, Loss: {loss.item()}")

### 2-Layered

In [65]:
class BrainSync_2(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.gat_layer1 = GATLayer(in_features, 4)
        self.gat_layer2 = GATLayer(4, out_features)
        self.lin_layer = torch.nn.Linear(out_features, 2)

    def forward(self, X, edge_index):
        res1 = self.gat_layer1(X, edge_index)
        res2 = self.gat_layer2(res1, edge_index)
        mean = torch.mean(res2, dim=0)
        classes = self.lin_layer(mean)
        
        return classes

In [66]:
# sync_2layered = BrainSync_2(6,4)
# optimizer = torch.optim.Adam(sync_2layered.parameters(), lr = 0.01)
# X = torch.randn(64, 6)
# edge_index_final = torch.randint(0, 64, (2, 200))
# labels_final = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = sync_2layered(X, edge_index_final)
#     loss = criterion(out, labels_final)
#     loss.backward()
#     optimizer.step()

#     # if epoch % 10 == 0:
#     #     print(f"Epoch {epoch}, Loss: {loss.item()}")

### 8-Layered

In [67]:
class BrainSync_8(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        
        self.layer1 = GATLayer(in_features, 4)
        self.layer2 = GATLayer(4, 4)
        self.layer3 = GATLayer(4, 4)
        self.layer4 = GATLayer(4, 4)
        self.layer5 = GATLayer(4, 4)
        self.layer6 = GATLayer(4, 4)
        self.layer7 = GATLayer(4, 4)
        self.layer8 = GATLayer(4, out_features)

        self.proj = torch.nn.Linear(in_features, 4)
        self.lin_layer = torch.nn.Linear(out_features, 2)


    def forward(self, X, edge_index):
        res1 = self.layer1(X, edge_index)
        prev_proj = self.proj(X)
        res1 = res1 + prev_proj
        res2 = self.layer2(res1, edge_index)
        res2 = res2 + res1 + prev_proj
        res3 = self.layer3(res2, edge_index)
        res3 = res3 + res2 + prev_proj
        res4 = self.layer4(res3, edge_index)
        res4 = res4 + res3 + prev_proj
        res5 = self.layer5(res4, edge_index)
        res5 = res5 + res4 + prev_proj
        res6 = self.layer6(res5, edge_index)
        res6 = res6 + res5 + prev_proj
        res7 = self.layer7(res6, edge_index)
        res7 = res7 + res6 + prev_proj
        res8 = self.layer8(res7, edge_index)
        res8 = res8 + res7 + prev_proj

        mean = torch.mean(res8, dim=0)

        classes = self.lin_layer(mean)

        return classes

In [68]:
# gat_8layered = BrainSync_8(6,4)
# optimizer = torch.optim.Adam(gat_8layered.parameters(), lr=0.0001)

# X = torch.randn(64, 6)
# edge_index_final = torch.randint(0, 64, (2, 200))
# labels_final = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

# for epoch in range(100):
#     optimizer.zero_grad()
#     out = gat_8layered(X, edge_index_final)
#     print(out)
#     loss = criterion(out, labels_final)
#     print(labels_final)
#     loss.backward()
#     optimizer.step()

#     if epoch % 10 == 0:
#         print(f"Epoch {epoch}, Loss: {loss.item()}")

In [69]:
# with torch.no_grad():
#     X = torch.randn(64, 6)
#     h = X.clone()
#     for layer in [gat_8layered.layer1, gat_8layered.layer2, gat_8layered.layer3,
#                   gat_8layered.layer4, gat_8layered.layer5, gat_8layered.layer6,
#                   gat_8layered.layer7, gat_8layered.layer8]:
#         h = layer(h, edge_index_final)
#         print(f"Std across nodes: {h.std(dim=0).mean().item():.6f}")

In [70]:
test_layer = GATLayer(6, 4)
X_test = torch.randn(10, 6)
edge_test = torch.randint(0, 10, (2, 20))
out = test_layer(X_test, edge_test)
# print(out) 

In [71]:
gat_8layered_skip = BrainSync_8(6, 4)
optimizer = torch.optim.Adam(gat_8layered_skip.parameters(), lr=0.01)

In [72]:
X = torch.randn(64, 6)

edge_index_final_skip = torch.randint(0, 64, (2, 200))
labels_final_skip = torch.randint(0, 2, (1,)).squeeze() #for the graph not the features

for epoch in range(100):
    optimizer.zero_grad()
    out = gat_8layered_skip(X, edge_index_final_skip)
    loss = criterion(out, labels_final_skip)
    # print(f"out: {out} | loss: {loss}")
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0: 
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.6628285050392151
Epoch 10, Loss: 0.001167331007309258
Epoch 20, Loss: 5.0424259825376794e-05
Epoch 30, Loss: 1.1920858014491387e-05
Epoch 40, Loss: 6.318072337307967e-06
Epoch 50, Loss: 4.887569048150908e-06
Epoch 60, Loss: 4.410734163684538e-06
Epoch 70, Loss: 4.172316494077677e-06
Epoch 80, Loss: 4.0531076592742465e-06
Epoch 90, Loss: 3.933898824470816e-06


In [73]:
with torch.no_grad():
    X = torch.randn(64, 6)
    h = X.clone()
    proj_X = gat_8layered_skip.proj(X)
    
    h = gat_8layered_skip.layer1(h, edge_index_final_skip) + proj_X
    print(f"Layer 1: {h.std(dim=0).mean().item():.6f}")
    
    for i, layer in enumerate([gat_8layered_skip.layer2, gat_8layered_skip.layer3,
                                gat_8layered_skip.layer4, gat_8layered_skip.layer5,
                                gat_8layered_skip.layer6, gat_8layered_skip.layer7,
                                gat_8layered_skip.layer8]):
        prev = h.clone()
        h = layer(h, edge_index_final_skip) + prev
        print(f"Layer {i+2}: {h.std(dim=0).mean().item():.6f}")

Layer 1: 0.701488
Layer 2: 0.785310
Layer 3: 0.826202
Layer 4: 0.998725
Layer 5: 1.493093
Layer 6: 1.558722
Layer 7: 1.980187
Layer 8: 2.229008


## Working with Synthetic Data

### Creating an synthetic dataset similar to DEAP dataset

In [74]:
eeg_data_arr = np.random.uniform(-100, 100, size=(40,32,8064)) # Gives us the data for 40 trials, using 32 nodes with 8064 timepoints within the 63 sec of data
labels_arr = np.random.randint(1, 10, size=(40,2)) # Gives 40 labels (one for each trial) wth two columns given to arousal and arousal


In [75]:
# labels = pd.DataFrame({"Valence": labels_arr[:, 0], "Arousal": labels_arr[:, 1], "Anxiety Quadrant": np.where((labels_arr[:,0] < 5) & (labels_arr[:,1] > 5), 1, 0)})
# labels["Anxiety Quadrant"].value_counts()

On randomly getting the valence arousal values, we were getting an unbalanced class. To resolve this and check the credibility of the model, we manually balance the classes. 

In [76]:
import random

manual_labels = []
for i in range(20):
    manual_labels.append(0)
for i in range(20):
    manual_labels.append(1)

random.shuffle(manual_labels)

labels = pd.DataFrame({"Valence": labels_arr[:, 0], "Arousal": labels_arr[:, 1], "Anxiety Quadrant": manual_labels})
# labels

### PSD and band power calc

In [ ]:
bands = []

for i in range(40):
    for j in range(32):
        freq, psd = scipy.signal.welch(eeg_data_arr[i, j], fs=128)

        # Masks
        delta_mask = (freq >= 0.5) & (freq <= 4)
        theta_mask = (freq > 4) & (freq <= 8)
        alpha_mask = (freq > 8.0) & (freq <= 13.0)
        beta_mask = (freq > 13) & (freq <= 30)
        gamma_mask = (freq > 30) 

        # Band powers
        delta_band_power = psd[delta_mask].sum()
        theta_band_power = psd[theta_mask].sum()
        alpha_band_power = psd[alpha_mask].sum()
        beta_band_power = psd[beta_mask].sum()
        gamma_band_power = psd[gamma_mask].sum()

        band = [delta_band_power, theta_band_power, alpha_band_power, beta_band_power, gamma_band_power]
        bands.append(band)

eeg_power_data = np.array(bands).reshape(40, 32, 5)

eye_labels = np.random.randint(0, 2, size = (40, 32, 1))

eeg_final = np.concatenate((eeg_power_data, eye_labels), axis=2)
eeg_final.shape


(40, 32, 6)

In [ ]:
montage = mne.channels.make_standard_montage('standard_1020')

deap_channels = [
    'AF3', 'AF4', 'Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8',
    'T7', 'T8', 'FT7', 'FT8', 'TP7', 'TP8',
    'Cz', 'C3', 'C4',
    'Pz', 'P3', 'P4', 'P7', 'P8',
    'Oz', 'O1', 'O2',
    'FCz', 'FC3', 'FC4', 'CPz', 'CP3', 'CP4'
]


positions = np.array([montage.get_positions()['ch_pos'][ch] for ch in deap_channels])

print(len(positions))

def check_neighbors():
    source = []
    neighbors = []

    for electrode in range(32): # Running over each electrode's coordinates
        for neighbor in range(32):
            if neighbor != electrode:
                distance_from_neighbor = 0.0
                for coord in range(len(positions[electrode])): # Running over each coordinate of electrode and the comparison electrode
                    distance_from_neighbor += (positions[electrode][coord] - positions[neighbor][coord]) ** 2
                
                distance_from_neighbor = distance_from_neighbor ** 0.5

            else:
                continue

            # print(f"Distance between: {node_names[electrode]} and {node_names[neighbor]} is: {distance_from_neighbor}")

            if distance_from_neighbor < 0.06:
                # print(f"{deap_channels[electrode]} and {deap_channels[neighbor]} are neighbours")
                source.append(electrode)
                neighbors.append(neighbor)
            
            # else:
            #     print(f"{node_names[electrode]} and {node_names[neighbor]} are not neighbours")
            #     print(distance_from_neighbor)

    return torch.tensor([source, neighbors], dtype=torch.long)

edge_index = check_neighbors()
print(edge_index)
edge_index.shape


32
tensor([[ 0,  0,  0,  0,  1,  1,  2,  2,  2,  3,  3,  3,  4,  4,  4,  4,  5,  5,
          5,  5,  6,  6,  6,  6,  7,  7,  7,  7,  8,  8,  8,  9,  9,  9, 10, 10,
         10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15, 16, 16, 17, 17, 18, 18, 19,
         19, 20, 21, 21, 21, 22, 22, 22, 23, 23, 24, 24, 24, 25, 25, 25, 26, 26,
         27, 27, 28, 28, 29, 29, 30, 30, 31, 31],
        [ 2,  4,  5,  7,  3,  6,  0,  3,  7,  1,  2,  8,  0,  5,  6, 26,  0,  4,
          7, 27,  1,  4,  8, 28,  0,  2,  5, 11,  3,  6, 12, 11, 13, 21, 12, 14,
         22,  7,  9,  8, 10,  9, 21, 10, 22, 26, 29, 27, 30, 28, 31, 19, 29, 18,
         30, 31,  9, 13, 24, 10, 14, 25, 24, 25, 21, 23, 25, 22, 23, 24,  4, 15,
          5, 16,  6, 17, 15, 18, 16, 19, 17, 20]])


torch.Size([2, 82])

### Creating Data objects for PyG

In [ ]:

dataset = []

for i in range(40): 
    data = Data(x=torch.tensor(eeg_final[i], dtype=torch.float), edge_index=edge_index, y=torch.tensor(labels.iloc[i, 2], dtype=torch.long))
    dataset.append(data)

# dataset

### BrainSync execution

In [85]:
gat_deap = BrainSync_8(6, 4)
optimizer = torch.optim.Adam(gat_deap.parameters(), lr=0.03)


# Uncomment when running else wastes resources
# for epoch in range(100):
#     losses = 0
#     for data in dataset:
#         optimizer.zero_grad()
#         out = gat_deap(data.x, data.edge_index)
#         loss = criterion(out, data.y)
#         loss.backward()
#         optimizer.step()
#         losses += loss

    # if epoch % 10 == 0: 
    #     print(f"Epoch {epoch}, Loss: {losses/len(dataset)}")

## DASPS Dataset

### EDA

In [117]:
import scipy.io #Used to read .mat files
import mat73

labels = mat73.loadmat("../dasps_data/DASPS+HAM labels/DASPS+HAM labels.mat")
preprocess_data = mat73.loadmat("../dasps_data/preprocessed_DASPS_dataset/preprocessed_DASPS_dataset.mat")

for outer_key in labels:
    print(f"Outer Key: {outer_key}")
    for inner_key in labels[outer_key]:
        print(f"Inner Keys: {inner_key}")


for outer_key in preprocess_data:
    print(f"Outer Key: {outer_key}")
    for inner_key in preprocess_data[outer_key]:
        print(f"Inner Keys: {inner_key}")
        

Outer Key: Regim_datasub
Inner Keys: HAM
Inner Keys: label
Inner Keys: trial
Outer Key: DASPS_datasub
Inner Keys: Labels
Inner Keys: trials


#### EDA for regim dataset

In [152]:
regim_datasub = labels["Regim_datasub"]
# print(regim_datasub)

ham_score = regim_datasub["HAM"]

label = []
valence = []
arousal = []
for raw_label in regim_datasub["label"]:
    label.append(raw_label[0])
    valence.append(raw_label[0][0])
    arousal.append(raw_label[0][1])

trial = regim_datasub["trial"]

# print(len(label))
# print(label)

regim_df = pd.DataFrame({"HAM": ham_score, "label": label, "trial": trial, "Anxiety": np.where((ham_score >= 25), 1, 0), "Valence": valence,
                         "Arousal": arousal})
regim_df

,HAM,label,trial,Anxiety,Valence,Arousal
0,37.0,"[1.0, 8.0]","[[[-1.5291498, -1.3815739, -2.4833934, -7.2052...",1,1.0,8.0
1,37.0,"[1.0, 8.0]","[[[-13.753272, -23.18774, 5.098279, 15.387456,...",1,1.0,8.0
2,37.0,"[3.0, 8.0]","[[[-3.5860517, -8.71853, -3.0083878, 4.1039734...",1,3.0,8.0
3,37.0,"[3.0, 8.0]","[[[-2.2386038, 6.367345, 14.369148, 6.69579, 1...",1,3.0,8.0
4,37.0,"[1.0, 8.0]","[[[4.2351537, 2.3277357, -2.6977339, -8.192119...",1,1.0,8.0
...,...,...,...,...,...,...
271,14.0,"[1.0, 9.0]","[[[-10.993634, -2.862689, 16.424618, 8.997491,...",0,1.0,9.0
272,14.0,"[5.0, 1.0]","[[[2.1919472, 0.9266483, -4.1027284, -5.513344...",0,5.0,1.0
273,14.0,"[5.0, 1.0]","[[[0.94581026, 0.6222368, -0.1414873, 3.542898...",0,5.0,1.0
274,14.0,"[5.0, 3.0]","[[[7.68468, 0.7949513, 8.922763, 18.403004, 9....",0,5.0,3.0


In [153]:
## Finding number of anxiouos and non anxious people

print(len(ham_score))
anxious = 0

for i in ham_score:
    if i >= 25:
        anxious += 1
    
print(f"Anxious: {anxious//12}")
print(f"Not Anxious: {(len(ham_score) - anxious)//12}")

# The dataset has 12 trials for each of the 23 participants => 12 * 23 gives 276 values
# The ham score is the same across all 12 trials for each participant

276
Anxious: 14
Not Anxious: 9


In [154]:
print(len(label))
print(label)

## The label has an 2d array of size 276, 2 where we have 276 1d arrays containing the valence and arousal scores of each of the 12 trials of
## all 23 participants

276
[array([1., 8.]), array([1., 8.]), array([3., 8.]), array([3., 8.]), array([1., 8.]), array([1., 8.]), array([3., 5.]), array([3., 5.]), array([3., 6.]), array([3., 6.]), array([1., 7.]), array([1., 7.]), array([1., 8.]), array([1., 8.]), array([5., 5.]), array([5., 5.]), array([5., 7.]), array([5., 7.]), array([3., 1.]), array([3., 1.]), array([5., 7.]), array([5., 7.]), array([5., 8.]), array([5., 8.]), array([5., 3.]), array([5., 3.]), array([3., 1.]), array([3., 1.]), array([1., 6.]), array([1., 6.]), array([7., 1.]), array([7., 1.]), array([3., 5.]), array([3., 5.]), array([5., 5.]), array([5., 5.]), array([1., 8.]), array([1., 8.]), array([3., 1.]), array([3., 1.]), array([1., 7.]), array([1., 7.]), array([5., 1.]), array([5., 1.]), array([1., 6.]), array([1., 6.]), array([1., 8.]), array([1., 8.]), array([1., 8.]), array([1., 8.]), array([2., 8.]), array([2., 8.]), array([1., 7.]), array([1., 7.]), array([1., 8.]), array([1., 8.]), array([5., 5.]), array([5., 5.]), array([1.

In [165]:
print(f"Number of trials: {len(trial)}")
# print(trial)

print(f"No. of nodes of information per trial: {len(trial[0][0])}")
# print(trial[0][0])

print(f"No. of datapoints per node: {len(trial[0][0][0])}")
print(trial[0][0][0])

## This contains the information about all 276 trials that happened in terms of band powers
## Each of the 276 values contain 14 arrays with each array having values over 1920 datapoints

Number of trials: 276
No. of nodes of information per trial: 14
No. of datapoints per node: 1920
[ -1.5291498  -1.3815739  -2.4833934 ... -14.574729   12.814454
   8.959355 ]


#### EDA for dasps dataset

In [ ]:
dasps_datasub = preprocess_data["DASPS_datasub"]

labels_dasps = dasps_datasub["Labels"]
trials_dasps = dasps_datasub["trials"]

dasps_df = pd.DataFrame({"Labels": dasps_datasub["Labels"], "Trials": dasps_datasub["trials"]})
# dasps_df

,Labels,Trials
0,"[1.0, 8.0]","[[[-1.5291498, -1.3815739, -2.4833934, -7.2052..."
1,"[1.0, 8.0]","[[[-0.24963914, 1.7843469, -3.9510148, -6.6866..."
2,"[1.0, 8.0]","[[[3.7036242, -0.3529512, -7.4991274, -9.04492..."
3,"[1.0, 8.0]","[[[13.661541, 12.680455, 4.120638, 1.5551265, ..."
4,"[1.0, 8.0]","[[[4.379575, 2.6701324, -1.0133007, 0.45999724..."
...,...,...
4135,"[5.0, 3.0]","[[[-0.9667366, -3.9399273, -9.299178, -6.69390..."
4136,"[5.0, 3.0]","[[[1.7914469, 0.5905763, -1.3557637, -4.591057..."
4137,"[5.0, 3.0]","[[[-12.429446, -0.9506944, 15.367565, 10.96789..."
4138,"[5.0, 3.0]","[[[8.621865, 5.7553964, -0.03637005, 4.148871,..."


In [180]:
print(len(labels_dasps))
print(labels_dasps)

4140
[[array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(1.), array(8.)], [array(3.), array(8.)], [array(3.), array(8.)], [array(3.), array(8.)], [array(3.), array(8.)], [array(3.), array(8.)], [array(3.), array(8.)], [array(3.), array(8.)], [array(3.), array(8.)], [array(3.), array(8.)], [array(3.), array(8.)], [array(3.), array(8.)], [array(3.)

In [ ]:
data = []

for i in range(1, 10):
    data_point = mat73.loadmat(f"../dasps_data/Preprocessed data mat/S0{i}preprocessed.mat")
    data.append(data_point)

for i in range(10, 24):
    data_point = mat73.loadmat(f"../dasps_data/Preprocessed data mat/S{i}preprocessed.mat")
    data.append(data_point)

# for outer_key in data:
#     print(outer_key)
#     for inner_key in outer_key:
#         print(inner_key)

## Not required as all relevant data already in regim

### Power band calc and data object generation

In [ ]:
band_dasps = []

for i in range(276):
    for j in range(14):
        freq, psd = scipy.signal.welch(trial[i][0][j], fs = 128)

        # Masks
        delta_mask = (freq >= 0.5) & (freq <= 4)
        theta_mask = (freq > 4) & (freq <= 8)
        alpha_mask = (freq > 8.0) & (freq <= 13.0)
        beta_mask = (freq > 13) & (freq <= 30)
        gamma_mask = (freq > 30) 

        # Band powers
        delta_band_power = psd[delta_mask].sum()
        theta_band_power = psd[theta_mask].sum()
        alpha_band_power = psd[alpha_mask].sum()
        beta_band_power = psd[beta_mask].sum()
        gamma_band_power = psd[gamma_mask].sum()

        band = [delta_band_power, theta_band_power, alpha_band_power, beta_band_power, gamma_band_power]
        band_dasps.append(band)

# print((np.array(band_dasps)).shape)

final_psd_data = np.array(band_dasps).reshape(276, 14, 5)

# final_psd_data.shape

(276, 14, 5)

In [203]:
dasps_channels = ['AF3', 'AF4', 'F3', 'F4', 'FC5', 'FC6', 'F7', 'F8', 'T7', 'T8', 'P7', 'P8', 'O1', 'O2']
positions_dasps = np.array([montage.get_positions()["ch_pos"][ch] for ch in dasps_channels])

def check_neighbors():
    source = []
    neighbors = []

    for electrode in range(14): # Running over each electrode's coordinates
        for neighbor in range(14):
            if neighbor != electrode:
                distance_from_neighbor = 0.0
                for coord in range(len(positions_dasps[electrode])): # Running over each coordinate of electrode and the comparison electrode
                    distance_from_neighbor += (positions_dasps[electrode][coord] - positions_dasps[neighbor][coord]) ** 2
                
                distance_from_neighbor = distance_from_neighbor ** 0.5

            else:
                continue

            if distance_from_neighbor < 0.07:
                # print(f"{deap_channels[electrode]} and {deap_channels[neighbor]} are neighbours")
                source.append(electrode)
                neighbors.append(neighbor)

    return torch.tensor([source, neighbors], dtype=torch.long)

edge_index_dasps = check_neighbors()
# print(edge_index_dasps)
# print(len(edge_index_dasps[0]))

In [215]:
dataset_dasps = []

for i in range(276):
    data = Data(x  = torch.tensor(final_psd_data[i], dtype=torch.float), edge_index = edge_index_dasps, 
                y = torch.tensor(regim_df['Anxiety'][i], dtype=torch.long))
    
    dataset_dasps.append(data)

dataset_dasps

[Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14, 5], edge_index=[2, 40], y=1),
 Data(x=[14

### Execution

In [230]:
gat_dasps = BrainSync_8(5, 4)
optimizer_dasps = torch.optim.Adam(gat_dasps.parameters(), lr = 0.001)
preds = []
ground_truth = []

for epoch in range(100):
    losses = 0

    for data in dataset_dasps:
        optimizer_dasps.zero_grad()
        out = gat_dasps(data.x, data.edge_index)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer_dasps.step()
        losses += loss

    if epoch % 10 == 0: 
        
        print(f"Epoch: {epoch} => Loss: {losses/len(dataset_dasps)}")

with torch.no_grad():
    for data in dataset_dasps:
        out = gat_dasps(data.x, data.edge_index)
        preds.append(torch.argmax(out))
        ground_truth.append(data.y.int())

Epoch: 0 => Loss: 61.95323944091797
Epoch: 10 => Loss: 2.2842624187469482
Epoch: 20 => Loss: 0.8886333703994751
Epoch: 30 => Loss: 1.0236307382583618
Epoch: 40 => Loss: 0.7390139698982239
Epoch: 50 => Loss: 0.7172719240188599
Epoch: 60 => Loss: 0.7113388776779175
Epoch: 70 => Loss: 0.8387417793273926
Epoch: 80 => Loss: 0.7043047547340393
Epoch: 90 => Loss: 0.7206211090087891


In [231]:
true_pos = 0
false_pos = 0
true_neg = 0
false_neg = 0

for i in range(len(preds)):
    if preds[i] == 0:
        if preds[i] == ground_truth[i]:
            true_neg += 1
        else:
            false_neg += 1
    if preds[i] == 1:
        if preds[i] == ground_truth[i]:
            true_pos += 1
        else: false_pos += 1
    

print(f"True Positive: {true_pos}")
print(f"False Positive: {false_pos}")
print(f"True Negative: {true_neg}")
print(f"False Negative: {false_neg}")

total = true_pos + false_pos + true_neg + false_neg

predicted_positive = true_pos + false_pos
actual_positive = true_pos + false_neg
predicted_negative = true_neg + false_neg
actual_negative = true_neg + false_pos

accuracy = (true_pos + true_neg) / (total)
precision = (true_pos) / (predicted_positive)
recall = (true_pos) / (actual_positive)
f1_score = (2 * precision * recall) / (precision + recall)

print(f1_score)

True Positive: 139
False Positive: 82
True Negative: 26
False Negative: 29
0.7146529562982005
